In [4]:
!pip install entsoe-py requests pandas
!pip install matplotlib seaborn scikit-learn
import pandas as pd
import requests
import time
from entsoe import EntsoePandasClient

import os
from dotenv import load_dotenv

import pandas as pd
import requests
import time
from entsoe import EntsoePandasClient


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# import pandas as pd
# import requests

# # Hämta t.ex. ett helt år (2024) från Nord Pool-spegeln
# # Format: https://www.elprisetjustnu.se/api/v1/prices/{år}/{månad}-{dag}_SE3.json
# dates = pd.date_range("2024-01-01", "2024-01-05")  # Ändra spann efter behov
# all_prices = []

# for date in dates:
#     url = f"https://www.elprisetjustnu.se/api/v1/prices/{date.strftime('%Y/%m-%d')}_SE3.json"
#     r = requests.get(url)
#     if r.status_code == 200:
#         all_prices.extend(r.json())

# df_prices = pd.DataFrame(all_prices)
# print(df_prices)
# df_prices = df_prices[["time_start", "EUR_per_kWh"]].rename(
#     columns={"time_start": "timestamp", "EUR_per_kWh": "spot_price_eur_kwh"}
# )
# df_prices["spot_price_eur_mwh"] = df_prices["spot_price_eur_kwh"] * 1000
# df_prices["timestamp"] = pd.to_datetime(df_prices["timestamp"])

# #print(df_prices.head())



In [9]:
# 1. Läs in variabler från .env
load_dotenv()

# 2. Hämta nyckeln
API_KEY = os.getenv("ENTSOE_API_KEY")
print(API_KEY)

746b47-0403-480a-b87d-41f1ec766a1c


In [10]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
token = os.getenv("ENTSOE_API_KEY", "").strip()

print(f"Längd: {len(token)} (ska vara 36)")
print(f"Sträng: {token}")

Längd: 36 (ska vara 36)
Sträng: 77f4497b-e959-453a-a522-2ce54c508ea0


In [11]:
import requests

url = "https://web-api.tp.entsoe.eu/api"
params = {
    "securityToken": API_KEY,
    "documentType": "A44",
    "in_Domain": "10Y1001A1001A44P",  # SE1
    "out_Domain": "10Y1001A1001A44P",
    "periodStart": "202501010000",
    "periodEnd": "202501012300",
}

r = requests.get(url, params=params)
print(f"Statuskod: {r.status_code}")

if r.status_code == 200:
    print("Token fungerar perfekt!")
elif r.status_code == 401:
    print(
        "Servern nekar nyckeln: Kontrollera att du klickade 'Save' på ENTSO-E-profilen."
    )
else:
    print(r.text)

Statuskod: 401
Servern nekar nyckeln: Kontrollera att du klickade 'Save' på ENTSO-E-profilen.


In [12]:
import os
from dotenv import load_dotenv

# Tvinga inläsning även om variabeln redan finns i minnet
load_dotenv(override=True)

api_key = os.getenv("ENTSOE_API_KEY", "").strip().strip('"').strip("'")

print(f"Längd på token: {len(api_key)} tecken (ska vara exakt 36)")
print(f"Börjar med:     {api_key[:6]}...")
print(f"Slutar med:     ...{api_key[-6:]}")
print(f"Innehåller bindestreck: {api_key.count('-')} st (ska vara 4)")

Längd på token: 36 tecken (ska vara exakt 36)
Börjar med:     77f449...
Slutar med:     ...508ea0
Innehåller bindestreck: 4 st (ska vara 4)


In [ ]:
# ==========================================
# DEL 1: HÄMTA ELPRISER FRÅN ENTSO-E
# ==========================================
import os
import time
import pandas as pd
from dotenv import load_dotenv
from entsoe import EntsoePandasClient

# 1. Konfiguration
load_dotenv()
API_KEY = os.getenv("ENTSOE_API_KEY")
if not API_KEY:
    raise ValueError("Hittade ingen ENTSOE_API_KEY i din .env-fil!")

ZONES = {"SE1": "SE_1", "SE2": "SE_2", "SE3": "SE_3", "SE4": "SE_4"}

START_DATE = "2025-01-01"
END_DATE = "2026-01-01"
CACHE_DIR = "cache_entsoe"

os.makedirs(CACHE_DIR, exist_ok=True)
client = EntsoePandasClient(api_key=API_KEY.strip())

# 2. Hämta zonvis i 3-månadersblock
date_range = pd.date_range(start=START_DATE, end=END_DATE, freq="3MS", tz="UTC")
zone_price_dfs = []

for zone_name, zone_code in ZONES.items():
    print(f"\nBehandlar elpriser för {zone_name}...")
    zone_blocks = []

    for i in range(len(date_range) - 1):
        start_ts = date_range[i]
        end_ts = date_range[i + 1]

        cache_file = os.path.join(
            CACHE_DIR,
            f"price_{zone_name}_{start_ts.strftime('%Y%m%d')}_{end_ts.strftime('%Y%m%d')}.csv",
        )

        # Läs från lokal cache om den redan finns
        if os.path.exists(cache_file):
            df_cached = pd.read_csv(cache_file)
            df_cached["timestamp"] = pd.to_datetime(df_cached["timestamp"])
            zone_blocks.append(df_cached)
            print(
                f"  [{zone_name}] Cache hittad: {start_ts.strftime('%Y-%m')} till {end_ts.strftime('%Y-%m')}"
            )
            continue

        # Hämta från API med upp till 3 försök vid timeout
        success = False
        for attempt in range(1, 4):
            try:
                print(
                    f"  [{zone_name}] Hämtar {start_ts.strftime('%Y-%m')} till {end_ts.strftime('%Y-%m')} (försök {attempt}/3)..."
                )
                series = client.query_day_ahead_prices(
                    zone_code, start=start_ts, end=end_ts
                )
                df_block = series.reset_index()
                df_block.columns = ["timestamp", f"price_{zone_name.lower()}_eur_mwh"]
                df_block["timestamp"] = pd.to_datetime(
                    df_block["timestamp"]
                ).dt.tz_convert("UTC")

                df_block.to_csv(cache_file, index=False)
                zone_blocks.append(df_block)
                success = True
                break
            except Exception as e:
                print(f"     Varning: {e}")
                time.sleep(2 * attempt)
            time.sleep(0.3)

        if not success:
            print(f"  Kunde inte hämta {zone_name} för {start_ts.strftime('%Y-%m')}")

    if zone_blocks:
        df_zone = pd.concat(zone_blocks, ignore_index=True).drop_duplicates(
            subset=["timestamp"]
        )
        zone_price_dfs.append(df_zone)

# 3. Slå ihop alla zonpriser och spara
df_all_prices = zone_price_dfs[0]
for df_p in zone_price_dfs[1:]:
    df_all_prices = pd.merge(df_all_prices, df_p, on="timestamp", how="outer")

df_all_prices = df_all_prices.sort_values(by="timestamp").reset_index(drop=True)
df_all_prices.to_csv("prices_all_zones.csv", index=False)

print("\n--- DEL 1 KLAR ---")
print(f"Sparade {len(df_all_prices)} rader till 'prices_all_zones.csv'")
display(df_all_prices.head())


Behandlar elpriser för SE1...
  [SE1] Hämtar 2025-01 till 2025-04 (försök 1/3)...
     Varning: 401 Client Error: Unauthorized for url: https://web-api.tp.entsoe.eu/api?documentType=A44&in_Domain=10Y1001A1001A44P&out_Domain=10Y1001A1001A44P&offset=0&contract_MarketAgreement.type=A01&securityToken=746b47-0403-480a-b87d-41f1ec766a1c&periodStart=202412310000&periodEnd=202504020000
  [SE1] Hämtar 2025-01 till 2025-04 (försök 2/3)...
     Varning: 401 Client Error: Unauthorized for url: https://web-api.tp.entsoe.eu/api?documentType=A44&in_Domain=10Y1001A1001A44P&out_Domain=10Y1001A1001A44P&offset=0&contract_MarketAgreement.type=A01&securityToken=746b47-0403-480a-b87d-41f1ec766a1c&periodStart=202412310000&periodEnd=202504020000
  [SE1] Hämtar 2025-01 till 2025-04 (försök 3/3)...
     Varning: 401 Client Error: Unauthorized for url: https://web-api.tp.entsoe.eu/api?documentType=A44&in_Domain=10Y1001A1001A44P&out_Domain=10Y1001A1001A44P&offset=0&contract_MarketAgreement.type=A01&securityToken

In [ ]:
# ==========================================
# DEL 2: HÄMTA VÄDER & SLÅ IHOP DATASET
# ==========================================
import time
import requests
import pandas as pd

# 1. Konfiguration för väderkoordinater
ZONE_COORDS = {
    "SE1": {"lat": 65.5848, "lon": 22.1567, "city": "Luleå"},
    "SE2": {"lat": 62.3908, "lon": 17.3069, "city": "Sundsvall"},
    "SE3": {"lat": 59.3293, "lon": 18.0686, "city": "Stockholm"},
    "SE4": {"lat": 55.6050, "lon": 13.0038, "city": "Malmö"},
}

START_DATE = "2025-01-01"
END_DATE = "2025-12-31"  # Sista datumet för väderdatan

weather_url = "https://archive-api.open-meteo.com/v1/archive"
zone_weather_dfs = []

# 2. Hämta väder per zon
print("Hämtar väderdata per zon från Open-Meteo...")
for zone_name, conf in ZONE_COORDS.items():
    print(f"Hämtar väder för {zone_name} ({conf['city']})...")
    params = {
        "latitude": conf["lat"],
        "longitude": conf["lon"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": "temperature_2m,wind_speed_10m,rain",
        "timezone": "UTC",
    }

    res = requests.get(weather_url, params=params).json()
    if "hourly" not in res:
        raise ValueError(f"Fel från Open-Meteo för {zone_name}: {res}")

    hourly = res["hourly"]
    df_w = pd.DataFrame(
        {
            "timestamp": pd.to_datetime(hourly["time"]).dt.tz_localize("UTC"),
            f"temp_{zone_name.lower()}_c": hourly["temperature_2m"],
            f"wind_{zone_name.lower()}_kmh": hourly["wind_speed_10m"],
            f"rain_{zone_name.lower()}_mm": hourly["rain"],
        }
    )
    zone_weather_dfs.append(df_w)
    time.sleep(0.2)

# Slå ihop väder för alla zoner
df_all_weather = zone_weather_dfs[0]
for df_w in zone_weather_dfs[1:]:
    df_all_weather = pd.merge(df_all_weather, df_w, on="timestamp", how="inner")

print(f"Väderdata hämtad: {len(df_all_weather)} rader.")

# 3. Läs in prisdatan från Del 1 och slå ihop
print("\nLäser in priser och slår ihop dataseten...")
df_prices = pd.read_csv("prices_all_zones.csv")
df_prices["timestamp"] = pd.to_datetime(df_prices["timestamp"])

df_final = pd.merge(df_all_weather, df_prices, on="timestamp", how="inner")
df_final = df_final.sort_values(by="timestamp").reset_index(drop=True)
df_final["timestamp_local"] = df_final["timestamp"].dt.tz_convert("Europe/Stockholm")

# 4. Spara slutresultat
output_filename = "all_zones_weather_and_prices_2025.csv"
df_final.to_csv(output_filename, index=False)

print("\n--- DEL 2 KLAR ---")
print(f"Slutgiltig masterfil sparad till '{output_filename}' ({len(df_final)} rader).")
display(df_final.head())

Hämtar väderdata per zon från Open-Meteo...
Hämtar väder för SE1 (Luleå)...


AttributeError: 'DatetimeIndex' object has no attribute 'dt'

In [ ]:
import pandas as pd

# Läs in den sparade datan (tar bråkdelen av en sekund)
df = pd.read_csv("se3_weather_and_prices.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 1. Kolla saknade värden
print("Saknade värden per kolumn:")
print(df.isna().sum())

# 2. Snabb överblick av min/max och medel
print("\nStatistik:")
print(df[["temperature_c", "wind_speed_kmh", "rain_mm", "spot_price_eur_mwh"]].describe().round(2))

# 3. Snabb korrelationsmatris (se hur mycket vinden pressar priset!)
print("\nKorrelation mot elpris:")
print(df[["temperature_c", "wind_speed_kmh", "rain_mm", "spot_price_eur_mwh"]].corr()["spot_price_eur_mwh"].round(3))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 1. Läs in datan
df = pd.read_csv("se3_weather_and_prices.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 2. Rensa eventuella extrema spikar (t.ex. energikris-toppar > 500 EUR) 
# så att regressionslinjen inte förvrängs helt av enstaka extremvärden
clean_df = df[(df["spot_price_eur_mwh"] >= 0) & (df["spot_price_eur_mwh"] <= 300)].dropna(
    subset=["wind_speed_kmh", "spot_price_eur_mwh"]
)

X = clean_df[["wind_speed_kmh"]]
y = clean_df["spot_price_eur_mwh"]

# 3. Träna linjär regression
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

slope = model.coef_[0]
intercept = model.intercept_
r2 = r2_score(y, y_pred)

print(f"Lutning (koefficient): {slope:.3f} EUR/MWh per km/h vind")
print(f"Intercept: {intercept:.2f} EUR/MWh")
print(f"Förklaringsgrad (R²): {r2:.4f}")

# 4. Skapa visualisering
plt.figure(figsize=(10, 6), dpi=100)

# Eftersom 35 000 punkter blir en gröt kör vi låg alpha (transparens)
sns.regplot(
    data=clean_df.sample(min(5000, len(clean_df))), # Sampla t.ex. 5000 punkter för snabbare och snyggare rendering
    x="wind_speed_kmh",
    y="spot_price_eur_mwh",
    scatter_kws={"alpha": 0.15, "color": "#1f77b4", "s": 15},
    line_kws={"color": "red", "linewidth": 2, "label": f"Trend: y = {slope:.2f}x + {intercept:.1f} (R² = {r2:.3f})"}
)

plt.title("SE3 Elpris vs. Vindhastighet (Linjär Regression)", fontsize=14, fontweight="bold")
plt.xlabel("Vindhastighet vid 10m (km/h)", fontsize=12)
plt.ylabel("Spotpris (EUR/MWh)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Läs in data och extrahera tidsfunktioner
df = pd.read_csv("se3_weather_and_prices.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Säkerställ svensk lokal tid så att morgontoppen hamnar runt kl 07–09 och inte förskjuts av UTC
if df["timestamp"].dt.tz is None:
    df["timestamp"] = df["timestamp"].dt.tz_localize("UTC").dt.tz_convert("Europe/Stockholm")
else:
    df["timestamp"] = df["timestamp"].dt.tz_convert("Europe/Stockholm")

df["hour"] = df["timestamp"].dt.hour
df["day_name"] = df["timestamp"].dt.day_name()
df["day_of_week"] = df["timestamp"].dt.dayofweek  # 0 = Måndag, 6 = Söndag
df["is_weekend"] = df["day_of_week"].isin([5, 6]).map({True: "Helg", False: "Vardag"})

# Sortera dagarna i rätt ordning
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_labels_sv = ["Mån", "Tis", "Ons", "Tor", "Fre", "Lör", "Sön"]

# 2. Skapa figuren med två delgrafer
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.2, 1]})

# --- GRAF 1: Heatmap (Veckodag vs Timme) ---
pivot_table = df.pivot_table(
    index="day_name", 
    columns="hour", 
    values="spot_price_eur_mwh", 
    aggfunc="mean"
).reindex(day_order)

sns.heatmap(
    pivot_table, 
    cmap="YlOrRd", 
    cbar_kws={"label": "Medelpris (EUR/MWh)"}, 
    ax=ax1,
    annot=False
)
ax1.set_title("Snittpris: Veckodag vs Timme (SE3)", fontsize=13, fontweight="bold")
ax1.set_xlabel("Timme på dygnet (00–23)", fontsize=11)
ax1.set_ylabel("", fontsize=11)
ax1.set_yticklabels(day_labels_sv, rotation=0)

# --- GRAF 2: Dygnsprofil (Vardag vs Helg) ---
sns.lineplot(
    data=df, 
    x="hour", 
    y="spot_price_eur_mwh", 
    hue="is_weekend", 
    palette={"Vardag": "#d62728", "Helg": "#1f77b4"},
    linewidth=2.5,
    ax=ax2
)
ax2.set_title("Dygnsrytm: Vardag vs Helg", fontsize=13, fontweight="bold")
ax2.set_xlabel("Timme på dygnet (00–23)", fontsize=11)
ax2.set_ylabel("Spotpris (EUR/MWh)", fontsize=11)
ax2.set_xticks(range(0, 24, 2))
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(title="", frameon=True)

plt.tight_layout()
plt.show()